[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/C.%20Core%20Yard%20%26%20Land-Side%20Operations%20%28The%20Heart%20of%20the%20Terminal%29/22.%20The%20Internal%20Vehicle%20%28Terminal%20Truck%29%20Dispatching%20Problem/P22-Tier-3_executed.ipynb)

---


# 22. The Internal Vehicle (Terminal Truck) Dispatching Problem
## Tier 3: Dragonfly Algorithm

### Goal and scope
This notebook applies a population-based Dragonfly Algorithm to a small terminal truck dispatching instance. It represents each candidate by a continuous priority vector, decodes that vector into a dispatch order, and assigns containers sequentially to feasible trucks. The algorithm searches for a low-cost dispatch plan; it does not prove that the plan is globally optimal.

The input consists of containers, trucks, and directed travel times between locations. The outputs are a truck assignment for each feasible container, a weighted-completion-time fitness value, a schedule summary, and plots of the search history and decoded plan.

### Modeling assumptions
- A truck handles at most one container at a time. Each truck becomes available again when its assigned container is delivered; no separate loading, unloading, or return-to-depot time is modeled.
- Travel times are deterministic, expressed in discrete time periods, and defined for every pair of distinct locations used by the instance.
- A container's earliest pickup time constrains when its dispatch can start. Its latest delivery time constrains its completion time.
- Container priority is a positive weight in the objective. An unassigned container incurs a fixed penalty of 1,000 times its priority.
- Capacity is one container per truck at a time. The model does not include traffic, breakdowns, battery or fuel limits, or uncertainty.

### Solution method
A dragonfly is represented by a vector with one real-valued coordinate per container. Sorting the coordinates in descending order gives a priority order. In that order, the decoder selects the feasible truck with the earliest completion time. The population-based search changes these coordinates using separation, alignment, cohesion, attraction to the best candidate (food), and repulsion from the worst candidate (enemy).

The decoder is a constructive scheduling rule, not an exhaustive enumeration of all truck assignments. Different vectors can produce the same priority order and therefore the same decoded plan. The nominal number of possible container-to-truck mappings is $|T|^{|C|}$, but the Dragonfly Algorithm searches the continuous priority-vector representation rather than enumerating that mapping space.

### Mathematical model used for evaluation
For a decoded dispatch order, let $C$ be the set of containers, $T$ the set of trucks, and $U$ the set of unassigned containers. For each assigned container $c$, let $t_c$ be its completion time and $p_c$ its priority. The decoder enforces the time-window condition $t_c \leq d_c$, where $d_c$ is the latest delivery time. The evaluated cost is

$$
J = \sum_{c \in C \setminus U} p_c t_c + 1000\sum_{c \in U}p_c,
\qquad
f = \frac{1}{1+J}.
$$

Here, $J$ is the weighted completion cost plus the penalty for unassigned containers, and $f$ is fitness, which is larger for a lower-cost plan. The penalty makes infeasible-to-assign containers visible in the score rather than silently ignoring them.

### Dragonfly search update
For dragonfly $i$, let $x_i$ be its priority vector and $v_i$ its search-step vector. Neighbors are candidates whose Euclidean distance from $x_i$ is less than the neighborhood radius. The three neighborhood behaviors are

$$
S_i=-\sum_{j\in N_i}(x_j-x_i),\qquad
A_i=\frac{1}{|N_i|}\sum_{j\in N_i}v_j,\qquad
C_i=\frac{1}{|N_i|}\sum_{j\in N_i}x_j-x_i.
$$

When there are no neighbors, all three behaviors are zero. Let $x_{food}$ denote the best candidate and $x_{enemy}$ the worst candidate in the current population. The implementation updates velocity and position using

$$
v_i^{(k+1)}=w v_i^{(k)}+sS_i+aA_i+cC_i+2(x_{food}-x_i^{(k)})+2(x_i^{(k)}-x_{enemy}),
\qquad
x_i^{(k+1)}=\operatorname{clip}\bigl(x_i^{(k)}+0.1v_i^{(k+1)},0,1\bigr).
$$

The coefficients $w,s,a,c$ correspond to the inertia and behavior weights in the code; the food and enemy weights are also stored in the parameter dictionary. The position is clipped because each priority coordinate is defined on $[0,1]$. This is a stochastic metaheuristic: its best-so-far score is non-increasing in cost, but convergence to a global optimum is not guaranteed. With population size $P$, $K$ iterations, dimension $n=|C|$, and $P$ candidates, fitness evaluation includes decoding and truck selection, while pairwise diversity and neighborhood calculations require up to $O(KP^2n)$ arithmetic. The number of priority orders can grow as $n!$, so the continuous encoding does not make the underlying scheduling problem trivial.

The sections below define the dispatch procedure, construct the instance, run the search, inspect and validate its result, visualize the search history, and examine sensitivity to the existing algorithm parameters.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Fixing the random seed makes the stochastic search reproducible when run
# with the same NumPy and Python environment.
np.random.seed(42)

print("Initializing the Dragonfly Algorithm for terminal truck dispatching.")

In [ ]:
@dataclass
class Container:
    """A container requiring transport service."""

    id: str
    origin: str
    destination: str
    priority: int
    earliest_pickup: int
    latest_delivery: int


@dataclass
class Truck:
    """A terminal truck and its initial operating state."""

    id: str
    current_location: str
    available_time: int
    capacity: int = 1


class DragonflyDispatcher:
    """Search over priority vectors and decode them into truck schedules."""

    def __init__(
        self,
        containers: List[Container],
        trucks: List[Truck],
        travel_matrix: Dict[Tuple[str, str], int],
    ) -> None:
        self.containers = containers
        self.trucks = trucks
        self.travel_matrix = travel_matrix
        self.dimension = len(containers)

        if not self.containers:
            raise ValueError("At least one container is required.")
        if not self.trucks:
            raise ValueError("At least one truck is required.")
        if len({container.id for container in containers}) != len(containers):
            raise ValueError("Container identifiers must be unique.")
        if len({truck.id for truck in trucks}) != len(trucks):
            raise ValueError("Truck identifiers must be unique.")

        locations = {
            location
            for container in containers
            for location in (container.origin, container.destination)
        }
        locations.update(truck.current_location for truck in trucks)
        missing_routes = [
            (origin, destination)
            for origin in locations
            for destination in locations
            if origin != destination and (origin, destination) not in travel_matrix
        ]
        if missing_routes:
            raise ValueError(
                "Travel times must be provided for every directed route. "
                f"Missing routes include: {missing_routes[:5]}"
            )

        self.population_size = 30
        self.max_iterations = 200
        self.weights = {
            "separation": 2.0,
            "alignment": 2.0,
            "cohesion": 2.0,
            "food": 2.0,
            "enemy": 2.0,
        }
        self.inertia_weight = 0.9
        self.best_solution = None
        self.best_fitness = -1.0
        self.fitness_history = []
        self.diversity_history = []

    def _travel_time(self, origin: str, destination: str) -> int:
        """Return a defined route time instead of treating missing routes as zero."""
        if origin == destination:
            return 0
        return self.travel_matrix[(origin, destination)]

    def initialize_population(self) -> List[np.ndarray]:
        """Create random priority vectors with coordinates in the unit interval."""
        return [
            np.random.rand(self.dimension)
            for _ in range(self.population_size)
        ]

    def decode_solution(self, individual: np.ndarray) -> Dict[str, str]:
        """Decode a priority vector into feasible container-to-truck assignments.

        Containers are considered from highest to lowest vector coordinate. For
        each one, the decoder chooses the feasible truck with the earliest
        completion time. The returned dictionary preserves this dispatch order.
        """
        if len(individual) != self.dimension:
            raise ValueError("The priority vector must have one coordinate per container.")

        priorities = sorted(
            enumerate(individual), key=lambda item: item[1], reverse=True
        )
        assignments = {}
        truck_available = {
            truck.id: truck.available_time for truck in self.trucks
        }
        truck_locations = {
            truck.id: truck.current_location for truck in self.trucks
        }

        for index, _ in priorities:
            container = self.containers[index]
            best_truck = None
            best_completion_time = float("inf")

            for truck in self.trucks:
                pickup_time = self._travel_time(
                    truck_locations[truck.id], container.origin
                )
                transport_time = self._travel_time(
                    container.origin, container.destination
                )
                start_time = max(
                    truck_available[truck.id], container.earliest_pickup
                )
                completion_time = start_time + pickup_time + transport_time

                if (
                    completion_time <= container.latest_delivery
                    and completion_time < best_completion_time
                ):
                    best_truck = truck
                    best_completion_time = completion_time

            if best_truck is not None:
                assignments[container.id] = best_truck.id
                truck_available[best_truck.id] = best_completion_time
                truck_locations[best_truck.id] = container.destination

        return assignments

    def evaluate_fitness(self, individual: np.ndarray) -> float:
        """Return the inverse penalized weighted-completion cost.

        The schedule is evaluated in the exact dispatch order produced by the
        decoder, so each truck's location and availability evolve consistently.
        """
        assignments = self.decode_solution(individual)
        container_by_id = {container.id: container for container in self.containers}
        truck_available = {
            truck.id: truck.available_time for truck in self.trucks
        }
        truck_locations = {
            truck.id: truck.current_location for truck in self.trucks
        }
        total_cost = 0.0

        for container_id, truck_id in assignments.items():
            container = container_by_id[container_id]
            pickup_time = self._travel_time(
                truck_locations[truck_id], container.origin
            )
            transport_time = self._travel_time(
                container.origin, container.destination
            )
            start_time = max(
                truck_available[truck_id], container.earliest_pickup
            )
            completion_time = start_time + pickup_time + transport_time
            total_cost += container.priority * completion_time
            truck_available[truck_id] = completion_time
            truck_locations[truck_id] = container.destination

        assigned_ids = set(assignments)
        total_cost += sum(
            1000 * container.priority
            for container in self.containers
            if container.id not in assigned_ids
        )
        return 1.0 / (1.0 + total_cost)

    def calculate_behaviors(
        self,
        population: List[np.ndarray],
        velocities: List[np.ndarray],
    ) -> List[Tuple[np.ndarray, np.ndarray, np.ndarray]]:
        """Compute separation, alignment, and cohesion for each candidate.

        Separation moves away from nearby positions, alignment follows the
        neighbors' velocity vectors, and cohesion points toward their center.
        A candidate without neighbors receives zero for these three terms.
        """
        behaviors = []
        neighbor_radius = 0.5

        for index, individual in enumerate(population):
            separation = np.zeros(self.dimension)
            alignment = np.zeros(self.dimension)
            cohesion = np.zeros(self.dimension)
            neighbors = 0

            for other_index, other in enumerate(population):
                if index == other_index:
                    continue
                distance = np.linalg.norm(individual - other)
                if distance < neighbor_radius:
                    neighbors += 1
                    separation -= (other - individual) / (distance + 1e-6)
                    alignment += velocities[other_index]
                    cohesion += other

            if neighbors:
                alignment /= neighbors
                cohesion = cohesion / neighbors - individual

            behaviors.append((separation, alignment, cohesion))

        return behaviors

    def calculate_diversity(self, population: List[np.ndarray]) -> float:
        """Measure diversity as the mean Euclidean distance between pairs."""
        if len(population) <= 1:
            return 0.0

        total_distance = 0.0
        pair_count = 0
        for index in range(len(population)):
            for other_index in range(index + 1, len(population)):
                total_distance += np.linalg.norm(
                    population[index] - population[other_index]
                )
                pair_count += 1
        return total_distance / pair_count

    def optimize(self) -> Tuple[Dict[str, str], float]:
        """Run the Dragonfly Algorithm and return its best decoded plan."""
        print("Starting Dragonfly Algorithm optimization.")
        print(f"Population size: {self.population_size}")
        print(f"Maximum iterations: {self.max_iterations}")

        # Reset run-specific state so repeated calls start a fresh search.
        self.best_solution = None
        self.best_fitness = -1.0
        self.fitness_history = []
        self.diversity_history = []

        population = self.initialize_population()
        velocities = [np.zeros(self.dimension) for _ in population]
        start_time = time.time()

        for iteration in range(self.max_iterations):
            fitness_values = [
                self.evaluate_fitness(individual) for individual in population
            ]
            best_index = int(np.argmax(fitness_values))
            if fitness_values[best_index] > self.best_fitness:
                self.best_fitness = fitness_values[best_index]
                self.best_solution = population[best_index].copy()

            diversity = self.calculate_diversity(population)
            self.fitness_history.append(self.best_fitness)
            self.diversity_history.append(diversity)
            behaviors = self.calculate_behaviors(population, velocities)
            worst_index = int(np.argmin(fitness_values))

            for index in range(self.population_size):
                separation, alignment, cohesion = behaviors[index]
                food = self.weights["food"] * (
                    self.best_solution - population[index]
                )
                enemy = self.weights["enemy"] * (
                    population[index] - population[worst_index]
                )
                velocities[index] = (
                    self.inertia_weight * velocities[index]
                    + self.weights["separation"] * separation
                    + self.weights["alignment"] * alignment
                    + self.weights["cohesion"] * cohesion
                    + food
                    + enemy
                )
                population[index] = np.clip(
                    population[index] + 0.1 * velocities[index], 0.0, 1.0
                )

            if (iteration + 1) % 50 == 0:
                print(
                    f"Iteration {iteration + 1}/{self.max_iterations}: "
                    f"best fitness = {self.best_fitness:.6f}, "
                    f"diversity = {diversity:.4f}"
                )

        print(f"\nOptimization completed in {time.time() - start_time:.2f} seconds.")
        best_assignments = self.decode_solution(self.best_solution)
        return best_assignments, self.best_fitness


print("Dragonfly Algorithm components are ready.")

## 1. Define the dispatching instance

The travel matrix is directed: a route from location $a$ to $b$ is represented by the key $(a,b)$. The instance below provides travel times for every directed route between distinct locations. Container records give their origin, destination, priority weight, earliest pickup time, and latest delivery time. Truck records give their initial location and availability time.

In [ ]:
travel_times = {
    ("berth", "storage"): 2, ("storage", "berth"): 2,
    ("storage", "rail"): 3, ("rail", "storage"): 3,
    ("storage", "gate"): 2, ("gate", "storage"): 2,
    ("berth", "rail"): 4, ("rail", "berth"): 4,
    ("berth", "gate"): 3, ("gate", "berth"): 3,
    ("rail", "gate"): 3, ("gate", "rail"): 3,
    ("berth", "yard"): 1, ("yard", "berth"): 1,
    ("storage", "yard"): 1, ("yard", "storage"): 1,
    ("rail", "yard"): 2, ("yard", "rail"): 2,
    ("gate", "yard"): 1, ("yard", "gate"): 1,
}

containers = [
    Container("C1", "berth", "storage", 10, 0, 8),
    Container("C2", "storage", "rail", 15, 1, 7),
    Container("C3", "storage", "gate", 8, 0, 10),
    Container("C4", "berth", "gate", 12, 2, 9),
    Container("C5", "rail", "storage", 18, 3, 12),
    Container("C6", "gate", "berth", 6, 1, 6),
    Container("C7", "yard", "rail", 20, 0, 8),
    Container("C8", "berth", "yard", 14, 2, 11),
    Container("C9", "storage", "yard", 9, 1, 9),
    Container("C10", "rail", "gate", 16, 4, 13),
    Container("C11", "gate", "storage", 11, 0, 7),
    Container("C12", "yard", "berth", 13, 2, 10),
    Container("C13", "berth", "rail", 17, 1, 8),
    Container("C14", "storage", "gate", 7, 3, 12),
    Container("C15", "rail", "yard", 19, 0, 9),
]

trucks = [
    Truck("T1", "berth", 0),
    Truck("T2", "storage", 0),
    Truck("T3", "gate", 0),
    Truck("T4", "rail", 1),
    Truck("T5", "yard", 0),
]

locations = sorted(
    {container.origin for container in containers}
    | {container.destination for container in containers}
    | {truck.current_location for truck in trucks}
)
total_priority = sum(container.priority for container in containers)
average_time_window = sum(
    container.latest_delivery - container.earliest_pickup
    for container in containers
) / len(containers)
theoretical_assignment_count = len(trucks) ** len(containers)

print("Dispatching instance created.")
print(f"Containers: {len(containers)}")
print(f"Trucks: {len(trucks)}")
print(f"Locations: {locations}")
print(f"Directed travel connections: {len(travel_times)}")
print(f"Total container priority: {total_priority}")
print(f"Average time-window width: {average_time_window:.1f} time periods")
print(
    "Nominal truck-assignment mappings: "
    f"{theoretical_assignment_count:,}"
)
print(
    "The priority-vector decoder explores schedules through dispatch orders; "
    "it does not enumerate all nominal mappings."
)

## 2. Run the Dragonfly Algorithm

Each candidate is decoded and evaluated in every iteration. The best-so-far candidate is retained separately from the moving population, so a later population update cannot erase the best score encountered during the search. The returned fitness is the inverse of the penalized objective defined above; it is not itself a cost or a percentage.

In [ ]:
dragonfly_dispatcher = DragonflyDispatcher(containers, trucks, travel_times)

print("\n" + "=" * 72)
print("DRAGONFLY ALGORITHM OPTIMIZATION")
print("=" * 72)
best_assignments, best_fitness = dragonfly_dispatcher.optimize()

print("\nSearch result summary")
print(f"Best fitness: {best_fitness:.8f}")
print(f"Assigned containers: {len(best_assignments)}/{len(containers)}")
print(f"Assignment rate: {len(best_assignments) / len(containers):.1%}")

## 3. Inspect and validate the decoded plan

The displayed schedule follows the dispatch order used by the decoder and fitness function. For each assignment, the report recomputes pickup travel, transport travel, and completion time while updating the truck's location and availability. The objective includes both assigned-container completion costs and penalties for unassigned containers, so the reported objective can be checked directly against the returned fitness.

In [ ]:
container_by_id = {container.id: container for container in containers}
truck_by_id = {truck.id: truck for truck in trucks}
truck_available = {truck.id: truck.available_time for truck in trucks}
truck_locations = {truck.id: truck.current_location for truck in trucks}
truck_utilization = {truck.id: 0 for truck in trucks}
assignment_details = []
assigned_completion_cost = 0.0

for container_id, truck_id in best_assignments.items():
    container = container_by_id[container_id]
    truck = truck_by_id[truck_id]
    pickup_time = dragonfly_dispatcher._travel_time(
        truck_locations[truck_id], container.origin
    )
    transport_time = dragonfly_dispatcher._travel_time(
        container.origin, container.destination
    )
    start_time = max(truck_available[truck_id], container.earliest_pickup)
    completion_time = start_time + pickup_time + transport_time
    container_cost = container.priority * completion_time

    assert completion_time <= container.latest_delivery
    assigned_completion_cost += container_cost
    truck_utilization[truck_id] += 1
    assignment_details.append(
        {
            "Container": container_id,
            "Truck": truck_id,
            "Route": f"{container.origin} to {container.destination}",
            "Priority": container.priority,
            "Start time": start_time,
            "Pickup travel": pickup_time,
            "Transport time": transport_time,
            "Completion time": completion_time,
            "Weighted completion cost": container_cost,
        }
    )
    truck_available[truck_id] = completion_time
    truck_locations[truck_id] = container.destination

assignment_df = pd.DataFrame(assignment_details)
unassigned_containers = [
    container for container in containers
    if container.id not in best_assignments
]
unassigned_penalty = sum(
    1000 * container.priority for container in unassigned_containers
)
objective_value = assigned_completion_cost + unassigned_penalty
fitness_from_report = 1.0 / (1.0 + objective_value)

print("Dispatch schedule")
if assignment_df.empty:
    print("No feasible container assignments were produced.")
else:
    print(assignment_df.to_string(index=False))

print("\nSolution metrics")
print(f"Assigned weighted completion cost: {assigned_completion_cost:.2f}")
print(f"Unassigned-container penalty: {unassigned_penalty:.2f}")
print(f"Penalized objective value: {objective_value:.2f}")
print(f"Fitness recomputed from objective: {fitness_from_report:.8f}")
print(f"Fitness returned by search: {best_fitness:.8f}")
print(f"Assigned containers: {len(best_assignments)}/{len(containers)}")
if not assignment_df.empty:
    print(
        "Mean completion time: "
        f"{assignment_df['Completion time'].mean():.2f} time periods"
    )
    print(
        "Completion-time standard deviation: "
        f"{assignment_df['Completion time'].std(ddof=0):.2f} time periods"
    )

print("\nTruck utilization (assigned containers)")
for truck_id, count in truck_utilization.items():
    print(f"{truck_id}: {count}")

if unassigned_containers:
    print("\nUnassigned containers")
    for container in unassigned_containers:
        print(
            f"{container.id}: {container.origin} to {container.destination}; "
            f"priority {container.priority}; "
            f"time window {container.earliest_pickup} to "
            f"{container.latest_delivery}"
        )

assert np.isclose(fitness_from_report, best_fitness, rtol=1e-10, atol=1e-12)
assert set(best_assignments.values()).issubset(truck_by_id)

## 4. Visualize convergence, diversity, and the schedule

The best-fitness curve records the best score found up to each iteration; it is therefore a best-so-far measure. Because fitness is the inverse of a nonnegative penalized cost, an increase in fitness corresponds to a decrease in that cost. Mean pairwise distance describes how spread out the priority vectors are, not how different their decoded assignments must be. The schedule plot uses one color per truck and shows the interval from dispatch start through delivery completion.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fitness_axis, diversity_axis, priority_axis, schedule_axis = axes.flat

iterations = np.arange(1, len(dragonfly_dispatcher.fitness_history) + 1)
fitness_axis.plot(
    iterations,
    dragonfly_dispatcher.fitness_history,
    color="navy",
    linewidth=2,
    label="Best-so-far fitness",
)
fitness_axis.set_title("Best-so-far fitness")
fitness_axis.set_xlabel("Iteration")
fitness_axis.set_ylabel("Fitness (higher is better)")
fitness_axis.grid(True, alpha=0.3)
fitness_axis.legend()


diversity_axis.plot(
    iterations,
    dragonfly_dispatcher.diversity_history,
    color="darkred",
    linewidth=2,
    label="Mean pairwise distance",
)
diversity_axis.set_title("Population diversity")
diversity_axis.set_xlabel("Iteration")
diversity_axis.set_ylabel("Mean Euclidean distance")
diversity_axis.grid(True, alpha=0.3)
diversity_axis.legend()

container_labels = [container.id for container in containers]
priority_values = dragonfly_dispatcher.best_solution
priority_axis.bar(container_labels, priority_values, color="steelblue")
priority_axis.set_title("Priority-vector coordinates in the best candidate")
priority_axis.set_xlabel("Container")
priority_axis.set_ylabel("Priority-vector coordinate")
priority_axis.set_ylim(0, 1)
priority_axis.tick_params(axis="x", rotation=45)
priority_axis.grid(True, alpha=0.3, axis="y")

if assignment_df.empty:
    schedule_axis.text(
        0.5,
        0.5,
        "No assignments to display",
        ha="center",
        va="center",
        transform=schedule_axis.transAxes,
    )
    schedule_axis.set_axis_off()
else:
    truck_colors = {
        truck.id: plt.get_cmap("tab10")(index % 10)
        for index, truck in enumerate(trucks)
    }
    for row_index, row in assignment_df.iterrows():
        duration = row["Completion time"] - row["Start time"]
        schedule_axis.barh(
            row_index,
            duration,
            left=row["Start time"],
            height=0.65,
            color=truck_colors[row["Truck"]],
            label=row["Truck"],
        )
    handles = [
        plt.Line2D(
            [0], [0], color=truck_colors[truck.id], linewidth=6, label=truck.id
        )
        for truck in trucks
        if truck.id in set(assignment_df["Truck"])
    ]
    schedule_axis.legend(handles=handles, title="Truck")
    schedule_axis.set_yticks(range(len(assignment_df)))
    schedule_axis.set_yticklabels(assignment_df["Container"])
    schedule_axis.set_title("Dispatch-to-delivery intervals")
    schedule_axis.set_xlabel("Time (periods)")
    schedule_axis.set_ylabel("Container")
    schedule_axis.grid(True, alpha=0.3, axis="x")

fig.suptitle("Dragonfly search and decoded dispatch plan", fontsize=14)
fig.tight_layout()
plt.show()

## 5. Check the search result

These checks verify properties of the returned plan and connect the search score to the model definition. They do not establish global optimality. The decoder is a priority-based constructive heuristic inside a Dragonfly search, so solution quality depends on both the decoding rule and the stochastic search trajectory.

In [ ]:
assigned_ids = set(best_assignments)
assert assigned_ids.issubset({container.id for container in containers})
assert set(best_assignments.values()).issubset({truck.id for truck in trucks})
assert len(assignment_df) == len(best_assignments)
assert np.isclose(
    best_fitness,
    1.0 / (1.0 + objective_value),
    rtol=1e-10,
    atol=1e-12,
)
assert len(dragonfly_dispatcher.fitness_history) == dragonfly_dispatcher.max_iterations
assert len(dragonfly_dispatcher.diversity_history) == dragonfly_dispatcher.max_iterations

print("Plan checks passed.")
print("Every assigned container meets its delivery deadline.")
print("Reported fitness matches the penalized objective.")
print(
    "The search histories contain one best-fitness and diversity value "
    "per completed iteration."
)
print(
    "A high fitness indicates a low penalized cost for this model; it is not "
    "a proof of optimality or a guarantee of performance on other instances."
)

## 6. Parameter sensitivity

This experiment retains the same instance, encoding, decoder, fitness definition, and Dragonfly update. It varies only the population size, iteration limit, and inertia weight in the stated scenarios. Since the search is stochastic, a single run per configuration is descriptive rather than a statistical comparison. Runtime and fitness should be interpreted together: a longer run may use more computation without producing a better decoded plan.

In [ ]:
parameter_scenarios = [
    {
        "name": "Default",
        "population_size": 30,
        "max_iterations": 200,
        "inertia": 0.9,
    },
    {
        "name": "Large Population",
        "population_size": 50,
        "max_iterations": 200,
        "inertia": 0.9,
    },
    {
        "name": "More Iterations",
        "population_size": 30,
        "max_iterations": 300,
        "inertia": 0.9,
    },
    {
        "name": "Low Inertia",
        "population_size": 30,
        "max_iterations": 200,
        "inertia": 0.5,
    },
    {
        "name": "High Inertia",
        "population_size": 30,
        "max_iterations": 200,
        "inertia": 0.95,
    },
]

scenario_results = []
for scenario in parameter_scenarios:
    print(f"\nRunning configuration: {scenario['name']}")
    test_dispatcher = DragonflyDispatcher(containers, trucks, travel_times)
    test_dispatcher.population_size = scenario["population_size"]
    test_dispatcher.max_iterations = scenario["max_iterations"]
    test_dispatcher.inertia_weight = scenario["inertia"]

    start_time = time.time()
    test_assignments, test_fitness = test_dispatcher.optimize()
    elapsed_time = time.time() - start_time

    scenario_results.append(
        {
            "Configuration": scenario["name"],
            "Population": scenario["population_size"],
            "Iterations": scenario["max_iterations"],
            "Inertia": scenario["inertia"],
            "Assigned containers": len(test_assignments),
            "Assignment rate": len(test_assignments) / len(containers),
            "Fitness": test_fitness,
            "Execution time (seconds)": elapsed_time,
        }
    )

scenario_df = pd.DataFrame(scenario_results)
print("\nParameter sensitivity results")
print(scenario_df.to_string(index=False, formatters={
    "Assignment rate": "{:.1%}".format,
    "Fitness": "{:.8f}".format,
    "Execution time (seconds)": "{:.2f}".format,
}))

best_scenario_index = int(scenario_df["Fitness"].to_numpy().argmax())
best_scenario = scenario_df.iloc[best_scenario_index]
print("\nHighest-fitness configuration in this set of single runs")
print(f"Configuration: {best_scenario['Configuration']}")
print(f"Fitness: {best_scenario['Fitness']:.8f}")
print(f"Assigned containers: {best_scenario['Assigned containers']}/{len(containers)}")
print(
    "This identifies the best observed run in this experiment only; "
    "it does not establish a generally best parameter setting."
)

## 7. Interpretation and limitations

The algorithm's search coordinates are priorities, not direct truck choices. A candidate is therefore judged only after its vector has been decoded into a feasible dispatch order and truck assignments. This distinction matters: fitness describes the decoded plan, while vector distance and diversity describe the continuous search representation.

The objective values early completion of high-priority containers and penalizes every unassigned container. A plan with fewer unassigned containers is not automatically better unless the resulting penalized objective is lower; the objective is the quantity summarized by fitness. The greedy truck choice in the decoder selects the earliest feasible completion for the current container, but it does not reconsider earlier assignments. Consequently, the decoder and the metaheuristic together define the method's search behavior and its limitations.

The example is deterministic in its travel-time data but stochastic in its population initialization. The fixed NumPy seed supports reproducibility in a consistent software environment. The observed best candidate is a heuristic result, not a certificate of optimality. The model also omits operational effects such as congestion, loading and unloading durations, and uncertain travel times. Natural extensions include adding such constraints or testing additional instances while retaining the same priority-vector and Dragonfly search framework.